<a href="https://colab.research.google.com/github/JulianSantos-LATAMAI/ECON-5200/blob/main/lab_19/lab_ch19_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 19: Tree-Based Models — Random Forests
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 30 min Core + 15 min Extension + SHAP Deep Dive

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

---

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load data
# -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Part 1: Find the Bug — Model Comparison (10 min)

The following code trains three models and reports their performance.
**Something is wrong with how the comparison is set up.** Find it, fix it, explain.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains deliberate error)
# Step 2: Model comparison — find the bug
# -----------------------------------------------------------

tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train, y_train)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

# BUG IS HERE: RF is evaluated on TRAINING data, not test data
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print('=== Model Comparison ===')
print(f"Single Tree  \u2014 R\u00b2: {r2_score(y_test, tree.predict(X_test)):.4f}")
print(f"Ridge        \u2014 R\u00b2: {r2_score(y_test, ridge.predict(X_test)):.4f}")
print(f"Random Forest \u2014 R\u00b2: {r2_score(y_train, rf.predict(X_train)):.4f}")  # \u2190 WRONG: using training set
print()
print('Conclusion: Random Forest achieves R\u00b2 > 0.97! Far superior to alternatives.')

### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and error type)
  The RF is evaluated on y_train/x_train instead of y_test/x_test.

2. **Why is this dangerous?** (what misleading conclusion does it lead to?
  ) It concludes and inflated r^2 of 0.97
3. **Fix the code below** and report the correct R²
        R^2 is 0.8051

**Verification checkpoint:** After fixing, the RF Test R² should be between 0.78 and 0.83. If you get >0.95, you haven't found the bug.

4. **Which chapter concept does this error violate?** (hint: Ch 15)
  This violated the train/test principle

In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Fix the model comparison bug from Part 1
# -----------------------------------------------------------

# YOUR FIX HERE
print(f"Random Forest — R²: {r2_score(y_test, rf.predict(X_test)):.4f}")


## Part 2: Find the Methodological Flaw — Feature Importance (10 min)

The following analysis uses feature importance to make a **causal claim**.
The code runs correctly. The methodology is wrong. Find the flaw.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains methodological flaw)
# Step 3: Feature importance with flawed causal reasoning
# -----------------------------------------------------------

rf_correct = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf_correct.fit(X_train, y_train)

importance = pd.Series(rf_correct.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Feature Importance (MDI):')
print(importance.round(4))
print()
print('POLICY RECOMMENDATION:')
print(f'The top predictor is {importance.index[0]} (importance = {importance.iloc[0]:.3f}).')
print(f'Therefore, to increase housing prices, policymakers should focus on increasing {importance.index[0]}.')
print(f'The second most important lever is {importance.index[1]}.')

### YOUR DIAGNOSIS

1. **What is the methodological flaw?** (the code is correct — the reasoning is wrong).  MDI ranks features by predictive usefulness, not causal influence — the model finding MedInc useful for splitting nodes does not mean increasing MedInc will cause prices to rise. The code confuses correlation with causation.
2. **Why can't we use MDI for policy recommendations?** (connect to Ch 10 DAGs and Ch 15 prediction vs. explanation)
  MedInc is entangled with confounders like school quality and job access that simultaneously drive both income and prices, so we can't isolate its causal path. Ch. 15 draws a hard line between prediction and explanation — minimizing RMSE is a fundamentally different goal than estimating a causal effect.
3. **What would you need to make a causal claim?** (hint: Ch 24 DML)
A valid causal claim requires Double Machine Learning (Ch. 24 DML), which residualizes both MedInc and housing prices on all confounders before estimating the treatment effect. This removes confounding from both sides and isolates how much a one-unit increase in MedInc actually causes prices to change.
4. **Bonus:** MDI has a known statistical bias. What is it, and what alternative would you use?

**Verification checkpoint:** Your diagnosis should mention at least: (a) prediction ≠ causation, (b) confounding/omitted variables, (c) MDI bias toward high-cardinality features.

In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Run permutation importance and write a proper (non-causal)
# interpretation of the results
# -----------------------------------------------------------

# The code correctly runs MDI feature importance but then makes an invalid causal claim —
#prediction does not equal causation. MedInc may top the ranking because it is entangled with confounders
#like school quality and job access that simultaneously drive both income and prices, not because it causally
#determines them. Additionally, MDI is biased toward high-cardinality continuous features like MedInc, so its top
#ranking may be a statistical artifact rather than a reflection of true importance.



## Part 3: Hyperparameter Tuning + XGBoost Comparison (10 min)

Tune the RF, then compare against XGBoost (gradient boosting).

In [ ]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fill in the blanks
# Tune RF with GridSearchCV and compare with GBR
# -----------------------------------------------------------


from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# --- GridSearchCV on RandomForestRegressor (reduced grid for speed) ---
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, None],
    'max_features': ['sqrt'],
}

rf_cv = RandomForestRegressor(random_state=RANDOM_STATE)
grid_search = GridSearchCV(
    rf_cv, param_grid, cv=3, scoring='r2', n_jobs=-1
)
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_
print(f"Best params: {grid_search.best_params_}")

# --- Gradient Boosting ---
gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=5,
    learning_rate=0.1, random_state=RANDOM_STATE
)
gbr.fit(X_train, y_train)

# --- Results table ---
models = {
    'Ridge':      ridge,
    'RF Default': rf,
    'RF Tuned':   best_rf,
    'GBR':        gbr,
}

print(f"\n{'Model':<15} {'Test RMSE':>12} {'Test R²':>10}")
print("-" * 40)
for name, model in models.items():
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    print(f"{name:<15} {rmse:>12.4f} {r2:>10.4f}")

---

## Extension: SHAP Analysis (5200 depth — 15 min)

Use SHAP to explain individual predictions. Compare MDI ranking vs. SHAP ranking.

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 4: SHAP setup and TreeExplainer
# -----------------------------------------------------------

# Install SHAP if needed
# !pip install shap
import shap
# src/shap_utils.py
"""
Reusable SHAP utilities for tree-based model explanations.
"""
from __future__ import annotations
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator
from sklearn.inspection import permutation_importance


def explain_prediction(
    model: BaseEstimator,
    X: pd.DataFrame,
    idx: int,
) -> None:
    """
    Generate a SHAP waterfall plot for a single observation.

    Parameters
    ----------
    model : fitted sklearn tree-based model
    X     : feature DataFrame (test set)
    idx   : integer row index to explain
    """
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    explanation = shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X.iloc[idx],
        feature_names=X.columns.tolist(),
    )
    shap.plots.waterfall(explanation)


def global_importance(
    model: BaseEstimator,
    X: pd.DataFrame,
) -> None:
    """
    Generate a SHAP beeswarm plot showing global feature importance.

    Parameters
    ----------
    model : fitted sklearn tree-based model
    X     : feature DataFrame (test set)
    """
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    explanation = shap.Explanation(
        values=shap_values,
        base_values=explainer.expected_value,
        data=X,
        feature_names=X.columns.tolist(),
    )
    shap.plots.beeswarm(explanation)


def compare_importance(
    model: BaseEstimator,
    X: pd.DataFrame,
    y: pd.Series | np.ndarray,
) -> pd.DataFrame:
    """
    Side-by-side comparison of MDI vs SHAP mean absolute importance.

    Parameters
    ----------
    model : fitted RandomForestRegressor
    X     : feature DataFrame (test set)
    y     : true target values

    Returns
    -------
    pd.DataFrame with columns: feature, MDI, SHAP
    """
    # MDI importance
    mdi = pd.Series(model.feature_importances_, index=X.columns)

    # SHAP importance
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    shap_imp = pd.Series(
        np.abs(shap_values).mean(axis=0), index=X.columns
    )

    comparison = pd.DataFrame({
        'MDI':  mdi / mdi.sum(),       # normalize for fair comparison
        'SHAP': shap_imp / shap_imp.sum(),
    }).sort_values('SHAP', ascending=False)

    # Plot side by side
    comparison.plot(kind='bar', figsize=(10, 5), title='MDI vs SHAP Importance')
    plt.ylabel('Normalized Importance')
    plt.tight_layout()
    plt.savefig('figures/importance_comparison.png', dpi=150)
    plt.show()

    return comparison


if __name__ == "__main__":
    # Demonstration usage
    from sklearn.datasets import fetch_california_housing
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor

    data = fetch_california_housing()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    print("=== Waterfall for observation 0 ===")
    explain_prediction(model, X_test, idx=0)

    print("=== Global beeswarm ===")
    global_importance(model, X_test)

    print("=== MDI vs SHAP comparison ===")
    df = compare_importance(model, X_test, y_test)
    print(df)

### SHAP Interpretation (write as a .py module)

Create a reusable `shap_analysis.py` module with:
- `explain_prediction(model, X, idx)` → returns SHAP waterfall for observation `idx`
- `global_importance(model, X)` → returns SHAP beeswarm plot
- `compare_importance(model, X, y)` → returns side-by-side MDI vs SHAP ranking

Include docstrings and type hints. This is a portfolio artifact.

---
## AI-Assisted Expansion: SHAP Dashboard + Reusable Module

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over decision trees, random forests, hyperparameter tuning, feature importance, and SHAP explanations. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task (5200 — Advanced)
Build TWO artifacts:

**Artifact 1: `src/shap_utils.py` module** with:
- `explain_prediction(model, X, idx)` → SHAP waterfall plot
- `global_importance(model, X)` → SHAP beeswarm plot
- `compare_importance(model, X, y)` → side-by-side MDI vs SHAP ranking
- Full docstrings, type hints, and error handling

**Artifact 2: Interactive Streamlit app** that lets the user:
1. Adjust `n_estimators` (1-500) and `max_features` (1-8) with sliders
2. See SHAP waterfall + beeswarm plots update with each parameter change
3. Compare RF vs Ridge vs GBR performance as hyperparameters change
4. Toggle between MDI, permutation, and SHAP importance rankings

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# Copy the P.R.I.M.E. prompt above into Claude, then paste
# the generated code here. Run it and verify.
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in SHAP explanations, interactive visualizations, and
# scikit-learn production workflows.
#
# [Request] I just completed a diagnosis-first lab where I
# compared Decision Trees, Ridge, Random Forests, and Gradient
# Boosting on California Housing data. I fixed evaluation bugs,
# diagnosed causal overclaiming from MDI, tuned hyperparameters
# with GridSearchCV, and generated SHAP waterfall + beeswarm
# plots. Now I need TWO artifacts:
#
# 1. A reusable `src/shap_utils.py` module with three functions:
#    - explain_prediction(model, X, idx) -> SHAP waterfall
#    - global_importance(model, X) -> SHAP beeswarm
#    - compare_importance(model, X, y) -> MDI vs SHAP side-by-side
#    Include type hints, docstrings, and error handling.
#
# 2. An interactive Plotly dashboard (or Streamlit app) with
#    ipywidgets sliders for n_estimators (1-500) and max_features
#    (1-8). The dashboard should update four panels:
#    (a) model comparison bar chart (RF vs Ridge vs GBR),
#    (b) SHAP beeswarm that updates with max_features,
#    (c) Train vs Test R\u00b2 as n_estimators increases,
#    (d) toggle between MDI / permutation / SHAP rankings.
#
# [Iterate] Use plotly.graph_objects, ipywidgets, shap, numpy,
# sklearn. Use the same variable names: X_train, X_test,
# y_train, y_test, data.feature_names. Do not use deprecated
# Plotly or SHAP functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - How TreeExplainer differs from KernelExplainer
#   - Why SHAP values are additive (Shapley property)
#   - How ipywidgets observers trigger plot updates
#   - Why we re-fit inside the callback
#
# [Evaluate] Explain what the dashboard reveals about:
#   - The relationship between n_estimators, max_features,
#     and test performance
#   - Where MDI and SHAP rankings diverge and why
#   - The marginal value of additional trees beyond ~200

# PASTE AI-GENERATED CODE BELOW:

"""
shap_utils.py — Reusable SHAP explanation utilities for tree-based models.
"""
from __future__ import annotations
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator
from sklearn.inspection import permutation_importance


def explain_prediction(model: BaseEstimator, X: pd.DataFrame, idx: int) -> None:
    """
    Render a SHAP waterfall plot for a single observation.

    Parameters
    ----------
    model : fitted sklearn tree-based estimator
    X     : feature DataFrame (typically X_test)
    idx   : integer row index of the observation to explain
    """
    if idx >= len(X):
        raise IndexError(f"idx={idx} out of range for DataFrame with {len(X)} rows.")
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)
    explanation = shap.Explanation(
        values=shap_vals[idx],
        base_values=explainer.expected_value,
        data=X.iloc[idx],
        feature_names=X.columns.tolist(),
    )
    shap.plots.waterfall(explanation)


def global_importance(model: BaseEstimator, X: pd.DataFrame) -> None:
    """
    Render a SHAP beeswarm plot showing global feature importance.

    Parameters
    ----------
    model : fitted sklearn tree-based estimator
    X     : feature DataFrame (typically X_test)
    """
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)
    explanation = shap.Explanation(
        values=shap_vals,
        base_values=explainer.expected_value,
        data=X,
        feature_names=X.columns.tolist(),
    )
    shap.plots.beeswarm(explanation)


def compare_importance(
    model: BaseEstimator,
    X: pd.DataFrame,
    y: np.ndarray | pd.Series,
) -> pd.DataFrame:
    """
    Plot and return a side-by-side MDI vs SHAP importance comparison.

    Parameters
    ----------
    model : fitted RandomForestRegressor
    X     : feature DataFrame (typically X_test)
    y     : true target values

    Returns
    -------
    pd.DataFrame with normalized MDI and SHAP columns, sorted by SHAP.
    """
    mdi = pd.Series(model.feature_importances_, index=X.columns)
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)
    shap_imp = pd.Series(np.abs(shap_vals).mean(axis=0), index=X.columns)

    comparison = pd.DataFrame({
        'MDI':  mdi / mdi.sum(),
        'SHAP': shap_imp / shap_imp.sum(),
    }).sort_values('SHAP', ascending=False)

    comparison.plot(kind='bar', figsize=(10, 5),
                    title='MDI vs SHAP Importance (normalized)')
    plt.ylabel('Normalized Importance')
    plt.tight_layout()
    plt.savefig('figures/importance_comparison.png', dpi=150)
    plt.show()
    return comparison


if __name__ == "__main__":
    from sklearn.datasets import fetch_california_housing
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor

    data = fetch_california_housing()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    explain_prediction(model, X_test, idx=0)
    global_importance(model, X_test)
    df = compare_importance(model, X_test, y_test)
    print(df)

---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Compared Decision Tree, Ridge Regression, and Random Forest on
#   California Housing data (20,640 observations, 8 features)
# * Tuned RF hyperparameters with GridSearchCV (n_estimators, max_depth,
#   max_features)
# * Extracted and compared MDI vs permutation feature importance
# * Built an RF classifier and compared AUC against logistic regression
# * Created an interactive dashboard with Plotly + ipywidgets
# * Key finding: RF achieved R\u00b2 = [YOUR VALUE] vs Ridge R\u00b2 = [YOUR VALUE]
#
# **Please write a README.md entry including:**
# 1. Project Title: Tree-Based Models \u2014 Random Forests
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

### Push to GitHub

```bash
cd econ-lab-19-random-forests
git add notebooks/ figures/ README.md verification-log.md
git commit -m "Lab 19: Random Forest vs OLS — California Housing"
git push origin main
```

Submit your GitHub repo link on Canvas.